In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

# 1. Load Environment Variables
load_dotenv()

# 2. Define File Paths (Ensure these match your Antigravity folder structure)
files = {
    "policy": "../data/Academic-policy_Академическая политика 2025-2026 (англ).pdf",
    "booklet": "../data/Booklet KazNU 2025.pdf"
}

def process_university_docs(file_dict):
    all_chunks = []
    
    # Using RecursiveCharacterTextSplitter to maintain paragraph integrity
    # Important for your Dissertation: We use headers as separators
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=["\n\n", "\n", ".", "!", "?", " "]
    )

    for doc_type, path in file_dict.items():
        print(f"--- Processing {doc_type}: {os.path.basename(path)} ---")
        loader = PyPDFLoader(path)
        docs = loader.load()
        
        # Add metadata so the Agent knows which file the info came from
        for doc in docs:
            doc.metadata["source_type"] = doc_type
            doc.metadata["university"] = "KazNU"
        
        chunks = text_splitter.split_documents(docs)
        all_chunks.extend(chunks)
        print(f"Created {len(chunks)} chunks from {doc_type}.")
        
    return all_chunks

# 3. Execute Processing
processed_chunks = process_university_docs(files)

# 4. Data Verification (Show this to your Supervisor)
# Convert to DataFrame just to visualize the first few chunks
df = pd.DataFrame([
    {"Source": c.metadata['source_type'], "Content": c.page_content[:200] + "..."} 
    for c in processed_chunks
])

print("\n--- Initial Progress Preview ---")
print(df.head())

--- Processing policy: Academic-policy_Академическая политика 2025-2026 (англ).pdf ---
Created 158 chunks from policy.
--- Processing booklet: Booklet KazNU 2025.pdf ---
Created 42 chunks from booklet.

--- Initial Progress Preview ---
   Source                                            Content
0  policy  NON-PROFIT JOINT STOCK COMPANY \n«AL-FARABI KA...
1  policy  2 \n \n \n NJSC Al-Farabi Kazakh National Univ...
2  policy  6. EDUCATIONAL PROCESS PLANNING (ACADEMIC CALE...
3  policy  14. FINAL EXAMINATION ...........................
4  policy  3 \n \n \n NJSC Al-Farabi Kazakh National Univ...


In [ ]:
import pickle
with open('../data/processed_chunks.pkl', 'wb') as f:
    pickle.dump(processed_chunks, f)

: 